# Speech Emotion Recognition CNN Training

Notebook ini bertujuan untuk melatih model Convolutional Neural Network (CNN)
menggunakan dataset hasil audio augmentation.

Output notebook:

- ser_model_augmented.keras
- history
- training log

In [17]:
# ==========================================================
# IMPORT LIBRARY
# ==========================================================

import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import GlobalAveragePooling2D

from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.layers import (

    Conv2D,

    MaxPooling2D,

    Flatten,

    Dense,

    Dropout,

    BatchNormalization

)

from tensorflow.keras.callbacks import (

    EarlyStopping,

    ModelCheckpoint,

    ReduceLROnPlateau

)

from tensorflow.keras.optimizers import Adam

In [2]:
# ==========================================================
# CONFIGURATION
# ==========================================================

MODEL_NAME = "ser_model_augmented.keras"

TEST_SIZE = 0.20

RANDOM_STATE = 42

BATCH_SIZE = 32

EPOCHS = 50

LEARNING_RATE = 0.001

In [3]:
# ==========================================================
# LOAD DATASET
# ==========================================================

X = np.load(
    "X_dataset_aug.npy"
)

y = np.load(
    "y_dataset_aug.npy"
)

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [4]:
# ==========================================================
# DATASET INFORMATION
# ==========================================================

print(f"X Shape : {X.shape}")

print(f"y Shape : {y.shape}")

print(f"Total Samples : {len(y)}")

X Shape : (6884, 128, 215, 1)
y Shape : (6884,)
Total Samples : 6884


In [5]:
# ==========================================================
# LABEL DISTRIBUTION
# ==========================================================

label_names = {

    0: "Anger",

    1: "Sadness",

    2: "Neutral",

    3: "Happiness"

}

unique, counts = np.unique(
    y,
    return_counts=True
)

print("Dataset Distribution\n")

for label, total in zip(unique, counts):

    print(f"{label_names[label]:<12}: {total}")

Dataset Distribution

Anger       : 1600
Sadness     : 1600
Neutral     : 1844
Happiness   : 1840


In [6]:
# ==========================================================
# TRAIN TEST SPLIT
# ==========================================================

X_train, X_val, y_train, y_val = train_test_split(

    X,

    y,

    test_size=TEST_SIZE,

    random_state=RANDOM_STATE,

    stratify=y,

    shuffle=True

)

In [7]:
# ==========================================================
# SPLIT INFORMATION
# ==========================================================

print(f"Training Data : {X_train.shape}")

print(f"Validation Data : {X_val.shape}")

print(f"Training Label : {y_train.shape}")

print(f"Validation Label : {y_val.shape}")

Training Data : (5507, 128, 215, 1)
Validation Data : (1377, 128, 215, 1)
Training Label : (5507,)
Validation Label : (1377,)


In [8]:
# ==========================================================
# TRAINING DISTRIBUTION
# ==========================================================

print("Training Distribution\n")

unique, counts = np.unique(
    y_train,
    return_counts=True
)

for label, total in zip(unique, counts):

    print(f"{label_names[label]:<12}: {total}")

Training Distribution

Anger       : 1280
Sadness     : 1280
Neutral     : 1475
Happiness   : 1472


In [9]:
# ==========================================================
# VALIDATION DISTRIBUTION
# ==========================================================

print("Validation Distribution\n")

unique, counts = np.unique(
    y_val,
    return_counts=True
)

for label, total in zip(unique, counts):

    print(f"{label_names[label]:<12}: {total}")

Validation Distribution

Anger       : 320
Sadness     : 320
Neutral     : 369
Happiness   : 368


In [10]:
# ==========================================================
# BUILD CNN MODEL
# ==========================================================

model = Sequential()

# ==========================================================
# BLOCK 1
# ==========================================================

model.add(

    Conv2D(

        filters=32,

        kernel_size=(3,3),

        activation="relu",

        padding="same",

        input_shape=X_train.shape[1:]

    )

)

model.add(

    BatchNormalization()

)

model.add(

    MaxPooling2D(

        pool_size=(2,2)

    )

)

model.add(

    Dropout(0.25)

)

# ==========================================================
# BLOCK 2
# ==========================================================

model.add(

    Conv2D(

        filters=64,

        kernel_size=(3,3),

        activation="relu",

        padding="same"

    )

)

model.add(

    BatchNormalization()

)

model.add(

    MaxPooling2D(

        pool_size=(2,2)

    )

)

model.add(

    Dropout(0.30)

)

# ==========================================================
# BLOCK 3
# ==========================================================

model.add(

    Conv2D(

        filters=128,

        kernel_size=(3,3),

        activation="relu",

        padding="same"

    )

)

model.add(

    BatchNormalization()

)

model.add(

    MaxPooling2D(

        pool_size=(2,2)

    )

)

model.add(

    Dropout(0.35)

)

# ==========================================================
# CLASSIFIER
# ==========================================================

model.add(

    GlobalAveragePooling2D()

)

model.add(

    Dense(

        128,

        activation="relu"

    )

)

model.add(

    Dropout(0.50)

)

model.add(

    Dense(

        4,

        activation="softmax"

    )

)

C:\Users\hafiz\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
# ==========================================================
# MODEL SUMMARY
# ==========================================================

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 128, 215, 32)        │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 128, 215, 32)        │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 64, 107, 32)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64, 107, 32)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 64, 107, 64)         │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 64, 107, 64)         │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 32, 53, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 32, 53, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 32, 53, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 32, 53, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 16, 26, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 16, 26, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 128)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │          16,512 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 4)                   │             516 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 110,596 (432.02 KB)

 Trainable params: 110,148 (430.27 KB)

 Non-trainable params: 448 (1.75 KB)

In [12]:
# ==========================================================
# COMPILE MODEL
# ==========================================================

optimizer = Adam(

    learning_rate=LEARNING_RATE

)

model.compile(

    optimizer=optimizer,

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

print("Model compiled successfully.")

Model compiled successfully.


In [13]:
# ==========================================================
# EARLY STOPPING
# ==========================================================

early_stopping = EarlyStopping(

    monitor="val_loss",

    patience=8,

    restore_best_weights=True,

    verbose=1

)

In [14]:
# ==========================================================
# REDUCE LEARNING RATE
# ==========================================================

reduce_lr = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=3,

    min_lr=1e-6,

    verbose=1

)

In [15]:
# ==========================================================
# MODEL CHECKPOINT
# ==========================================================

checkpoint = ModelCheckpoint(

    MODEL_NAME,

    monitor="val_accuracy",

    save_best_only=True,

    mode="max",

    verbose=1

)

In [16]:
# ==========================================================
# CALLBACKS
# ==========================================================

callbacks = [

    early_stopping,

    reduce_lr,

    checkpoint

]

print("Callbacks created successfully.")

Callbacks created successfully.


In [18]:
# ==========================================================
# COMPUTE CLASS WEIGHT
# ==========================================================

classes = np.unique(y_train)

weights = compute_class_weight(

    class_weight="balanced",

    classes=classes,

    y=y_train

)

class_weights = dict(

    zip(

        classes,

        weights

    )

)

print("Class Weight\n")

for key, value in class_weights.items():

    print(

        f"{label_names[key]:<12}: {value:.4f}"

    )

Class Weight

Anger       : 1.0756
Sadness     : 1.0756
Neutral     : 0.9334
Happiness   : 0.9353


In [19]:
# ==========================================================
# TRAIN MODEL
# ==========================================================

history = model.fit(

    X_train,

    y_train,

    validation_data=(

        X_val,

        y_val

    ),

    epochs=EPOCHS,

    batch_size=BATCH_SIZE,

    callbacks=callbacks,

    class_weight=class_weights,

    verbose=1

)

Epoch 1/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 0s 582ms/step - accuracy: 0.2730 - loss: 1.4445
Epoch 1: val_accuracy improved from None to 0.26725, saving model to ser_model_augmented.keras

Epoch 1: finished saving model to ser_model_augmented.keras
173/173 ━━━━━━━━━━━━━━━━━━━━ 109s 608ms/step - accuracy: 0.3000 - loss: 1.3815 - val_accuracy: 0.2672 - val_loss: 1.5014 - learning_rate: 0.0010
Epoch 2/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 0s 577ms/step - accuracy: 0.3582 - loss: 1.3187
Epoch 2: val_accuracy improved from 0.26725 to 0.26797, saving model to ser_model_augmented.keras

Epoch 2: finished saving model to ser_model_augmented.keras
173/173 ━━━━━━━━━━━━━━━━━━━━ 104s 600ms/step - accuracy: 0.3750 - loss: 1.2997 - val_accuracy: 0.2680 - val_loss: 2.5493 - learning_rate: 0.0010
Epoch 3/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 0s 578ms/step - accuracy: 0.4191 - loss: 1.2509
Epoch 3: val_accuracy did not improve from 0.26797
173/173 ━━━━━━━━━━━━━━━━━━━━ 104s 600ms/step - accuracy: 0.4251 - loss: 1.2438 -

In [20]:
# ==========================================================
# SAVE VALIDATION DATASET
# ==========================================================

np.save(

    "X_validation.npy",

    X_val

)

np.save(

    "y_validation.npy",

    y_val

)

print("Validation dataset saved successfully.")

Validation dataset saved successfully.


In [21]:
# ==========================================================
# VALIDATION DATASET INFORMATION
# ==========================================================

print("Validation Feature :", X_val.shape)

print("Validation Label   :", y_val.shape)

Validation Feature : (1377, 128, 215, 1)
Validation Label   : (1377,)


In [22]:
# ==========================================================
# SAVE TRAINING HISTORY
# ==========================================================

import pickle

with open(

    "training_history.pkl",

    "wb"

) as file:

    pickle.dump(

        history.history,

        file

    )

print("Training history saved successfully.")

Training history saved successfully.
